# Obligatorio — Taller de Tecnologías 2 (ORT)
## Agentic RAG con LangGraph sobre tickets de soporte y papers

**Integrantes**

| Nombre | N° de estudiante |
|---|---|
| Martin Bernheim | _completar_ |
| Matías Iglesias | 310264 |

Agente construido **a mano** con `StateGraph` de LangGraph. No se usa `create_agent()`, `create_react_agent()`, `ToolNode` ni `tools_condition`. El agente:

1. Responde preguntas sobre el dataset de Kaggle **Multilingual Customer Support Tickets**.
2. Responde preguntas sobre los papers de la cátedra con chunking + RAG.
3. Tiene memoria de corto plazo (checkpointer por `thread_id`) y de largo plazo (store por `user_id`).
4. Soporta varios usuarios y conversaciones en paralelo.
5. Distingue small-talk, preguntas sobre documentos o el dataset, y preguntas generales, y admite cuando no tiene información.
6. Usa tool calling. Además del retrieval, tiene una tool de acción real: estadísticas con pandas.

### Índice
1. Setup y conexión a servicios
2. Base de conocimiento: papers
3. Base de conocimiento: tickets
4. Tools
5. El grafo (agente)
6. Memoria
7. Pruebas
8. Documentación

---
# Parte 1 — Setup y conexión a servicios

Stack (ver `docs/plan-arquitectura.md`):

| Componente | Elección |
|---|---|
| LLM de generación | `gpt-4o-mini` |
| LLM del router | `gpt-4.1-nano`: más barato y rápido, alcanza para clasificar |
| Embeddings | `text-embedding-3-small` (1536 dimensiones) |
| Base vectorial | Pinecone: un índice con dos namespaces, `papers` y `tickets` |
| Orquestación | LangGraph, con el grafo armado a mano |

Las API keys se leen de `.env`, que no se versiona. El notebook nunca imprime las keys.

In [ ]:
# Dependencias
# - En local: una sola vez, `pip install -r requirements.txt` en el .venv (esta celda no hace nada).
# - En Colab: esta celda instala todo sola.
# Cada vez que el notebook use una librería nueva, hay que agregarla acá Y en requirements.txt.
import sys
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    %pip install -q langgraph langchain-openai langchain-pinecone pinecone pandas pymupdf python-dotenv
else:
    print("Entorno local: dependencias desde requirements.txt")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# El notebook puede correr desde la raíz del repo o desde notebook/: buscamos la carpeta data/ hacia arriba.
# En Colab: clonar el repo o subir data/ al runtime antes de correr esta celda.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists()), Path.cwd())

# Keys: en local salen del .env; en Colab, de los Secrets (ícono de la llave en la barra lateral)
if EN_COLAB:
    from google.colab import userdata
    for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
        os.environ[key] = userdata.get(key)
else:
    load_dotenv(ROOT / ".env")

DATA_DIR    = ROOT / "data"
PAPERS_DIR  = DATA_DIR / "papers"
DATASET_CSV = DATA_DIR / "dataset" / "dataset-tickets-multi-lang3-4k.csv"

# Modelos y Pinecone
LLM_MODEL       = "gpt-4o-mini"
ROUTER_MODEL    = "gpt-4.1-nano"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM   = 1536
PINECONE_INDEX  = "obligatorio-tt2"
NS_PAPERS, NS_TICKETS = "papers", "tickets"

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    print(f"{key:18} {'OK' if os.getenv(key) else 'FALTA en .env'}")
print("Dataset encontrado:", DATASET_CSV.exists(), "| Papers:", len(list(PAPERS_DIR.glob("*.pdf"))))

### Prueba de conexión
Antes de construir nada, verificamos que cada servicio responda. Si la key de OpenAI está vencida o es inválida, la celda lo dice claramente en vez de cortar con un traceback.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import openai

def probar(nombre, fn):
    try:
        print(f"✅ {nombre}: {fn()}")
    except openai.AuthenticationError:
        print(f"❌ {nombre}: key de OpenAI inválida o vencida")
    except openai.PermissionDeniedError:
        print(f"❌ {nombre}: la key no tiene permiso para este modelo")
    except Exception as e:
        print(f"❌ {nombre}: {type(e).__name__}: {str(e)[:150]}")

probar(LLM_MODEL,       lambda: ChatOpenAI(model=LLM_MODEL, max_tokens=10).invoke("Decí solo: ok").content)
probar(ROUTER_MODEL,    lambda: ChatOpenAI(model=ROUTER_MODEL, max_tokens=10).invoke("Decí solo: ok").content)
probar(EMBEDDING_MODEL, lambda: f"vector de {len(OpenAIEmbeddings(model=EMBEDDING_MODEL).embed_query('hola'))} dimensiones")

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
indices = pc.list_indexes().names()
print("✅ Pinecone conectado. Índices existentes:", indices or "(ninguno)")
print(f"Índice del obligatorio '{PINECONE_INDEX}':", "ya existe" if PINECONE_INDEX in indices else "se crea en la Parte 2")

---
# Parte 2 — Base de conocimiento: papers

_Pendiente._

Pipeline: extracción → limpieza → chunking → embeddings → Pinecone (namespace `papers`).

- Usar PyMuPDF (`doc.get_toc()`) para hacer chunking por secciones y guardar `doc_id`, sección y página como metadata.
- Sacar las referencias bibliográficas antes de indexar (incluidas las del final de cada capítulo del libro).
- Limpiar el libro: guion U+2010 y encabezados de página tipo "Reliability | 9".
- Deduplicar por `doc_id`.
- Indexar una sola vez; en las corridas siguientes, solo conectarse al índice.

In [ ]:
# TODO

---
# Parte 3 — Base de conocimiento: tickets

_Pendiente._

- Cargar el CSV con pandas y limpiar nulos.
- Un ticket = un chunk (subject + body + answer).
- Metadata: `queue`, `priority`, `type`, `language`, `tags`.
- Indexar en el namespace `tickets`.

In [ ]:
# TODO

---
# Parte 4 — Tools

_Pendiente._

Funciones con `@tool` y docstrings claros, porque el LLM elige la tool leyendo la descripción.
- `buscar_tickets(query, filtros)`: búsqueda semántica.
- `estadisticas_tickets(...)`: conteos y filtros con pandas, porque los vectores no saben contar.
- `buscar_documentos(query, doc_id=None)`
- `resumir_documento(doc_id)`: trae abstract, introducción y conclusión por metadata.
- `guardar_memoria(hecho)` / `recuperar_memoria(query)`

Probar cada tool por separado antes de dársela al LLM.

In [ ]:
# TODO

---
# Parte 5 — El grafo (agente)

_Pendiente._

Grafo armado a mano, sin prebuilts:
- `State` que extiende `MessagesState` (ruta, memorias, iteraciones).
- Nodos: router → cargar_memoria → agente ⇄ ejecutar_tools → verificador → END.
- Ejecutor de tools y condición de corte escritos por nosotros, en lugar de `ToolNode` y `tools_condition`.
- Límite de iteraciones del loop.

In [ ]:
# TODO

---
# Parte 6 — Memoria

_Pendiente._

- Corto plazo: checkpointer por `thread_id` y sliding window con `trim_messages`.
- Largo plazo: store con namespace `(user_id, "memorias")`, compartido entre threads del mismo usuario.
- Demostrar memoria entre conversaciones y dos usuarios en paralelo.

In [ ]:
# TODO

---
# Parte 7 — Pruebas

_Pendiente._

Batería de preguntas:
- Dataset: semánticas y estadísticas.
- Cada paper.
- Small-talk.
- Preguntas generales.
- Multi-turno.
- Memoria de largo plazo.
- Preguntas trampa sin respuesta en el material.

In [ ]:
# TODO

---
# Parte 8 — Documentación

## 8.1 Arquitectura y justificación de decisiones técnicas
_Pendiente._

## 8.2 Estrategia de memoria
_Pendiente._

## 8.3 Prompts y prompt engineering
_Pendiente._

## 8.4 Desafíos y soluciones
_Pendiente._

## 8.5 Pruebas y resultados
_Pendiente._

## 8.6 Reflexión y experimentación opcional
_Pendiente._

## 8.7 Uso de IA generativa
_Pendiente: herramientas usadas y para qué (obligatorio)._